# 07 · MDP、Bellman 方程与回报

这一课把“驾驶”先压缩成一张纸上的有限 MDP。目标不是把表格游戏叫作自动驾驶，而是建立能手算、能断言、能解释的词汇：即时奖励 `r_t` 是一步发生了什么，回报 `G_t` 是从某时刻往后的折扣奖励之和，价值 `V(s)` 是策略下回报的期望，策略 `π(a|s)` 才是如何选动作。

先读代码前写下预测：`start` 选择 `short` 会得到多少回报？折扣系数 `γ=0.9` 时，`long` 为什么可能更好？有限 horizon 到达终点时是否需要猜一个未来价值？

<figure class="course-figure">
  <img src="../../assets/visuals/reinforcement-learning.png" width="1536" height="1024" style="max-width:100%;height:auto" alt="原理图：纸上 MDP 的短路线与长路线连接到驾驶中的 2 或 6 米每秒速度选择，转向保持几何控制">
  <a href="../../assets/visuals/reinforcement-learning.png">查看原图</a>
  <figcaption><strong>AI 原理图 · 手算/机制示意</strong> · 纸上 MDP（γ=0.9）与驾驶速度选择</figcaption>
</figure>

**因果链**：先用纸上 MDP 理解回报与价值；再在驾驶任务中学习 2/6 m/s 速度选择 → REINFORCE 更新策略。

**手算检查**：`γ=0.9` 时，`G(short)=2`，`G(long)=1+0.9×4`。哪条路线回报更高？
<details><summary>展开答案</summary><p>长路线回报为 4.6，高于短路线的 2。</p></details>

In [ ]:
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.rl_foundations import *

mdp = teaching_mdp()
print(mdp)

## 1. 手算一个确定性 MDP

`start --short/2--> terminal`，而 `start --long/1--> long --任意动作/4--> terminal`。因此：

`G(short)=2`，`G(long)=1+0.9×4=4.6`。

注意 4 是第二步的奖励，不是第一步的 `start` 价值；奖励由转移发生时产生。若把终点误算成一个普通状态，就会凭空多出未来奖励。

### 可编辑的奖励练习

下面先只改第二步奖励 `r`，其余状态和转移保持不动。长路线胜过短路线的阈值满足 `1 + γr = 2`，所以 `r = 1/γ`；当 `γ=0` 时只看当前奖励，short 的 2 大于 long 的 1。

In [ ]:
editable_second_step_reward = 4.0
gamma = 0.9
threshold = 1.0 / gamma
from dataclasses import replace
edited_mdp = replace(mdp, rewards={**mdp.rewards,
                     "long": {action: editable_second_step_reward for action in mdp.actions}})
edited_values = {s: 0.0 for s in mdp.states}
for _ in range(3):
    edited_values = bellman_optimality_update(edited_mdp, edited_values, gamma)
long_return = 1 + gamma * editable_second_step_reward
print({"threshold_r": threshold, "chosen_r": editable_second_step_reward,
       "short_return": 2.0, "long_return": long_return, "optimal_start_value": edited_values["start"]})
assert np.isclose(edited_values["start"], max(2.0, long_return))
assert np.isclose(1 + gamma * threshold, 2)
assert 2.0 > 1.0  # gamma=0: short wins immediately

In [ ]:
values = {s: 0.0 for s in mdp.states}
history = []
for _ in range(4):
    values = bellman_optimality_update(mdp, values, gamma=0.9)
    history.append(values.copy())
print(history)
assert np.isclose(history[-1]["start"], 4.6)
assert np.isclose(discounted_return([1, 4], 0.9), 4.6)

## 2. Bellman 更新和 Q-learning 的关系

最优 Bellman 更新把所有动作的候选值取最大：`V(s)←max_a [r+γ E V(s')]`。固定策略时把 `max` 换成策略的动作/期望。下面仍然是同步更新：右边只读上一轮的表。

Q-learning 不需要先知道转移表；它从一次实际样本 `(s,a,r,s')` 做增量更新：`Q←Q+α(target−Q)`，终止样本的 target 就是 `r`。这是学习估计，不是把一次样本冒充精确答案。

In [ ]:
print("one Q update:", q_learning_update(0.0, 1.0, 4.0, learning_rate=0.5, gamma=0.9))
assert np.isclose(q_learning_update(0, 2, 99, terminal=True), 0.2)
fixed = bellman_policy_update(mdp, {s: 0 for s in mdp.states}, {"start": "long", "long": "short"})
print("fixed-policy one-step values:", fixed)

## 3. reward、return、value、policy 的检查表

- reward：一步的结果，例如 `1` 或 `4`。
- return：一条已采样轨迹的数值，例如 `[1,4]` 的 `G_0=4.6`。
- value：在策略和环境随机性下，对 return 的期望；它不等于任意一条轨迹的 return。
- policy：给定状态的动作分布；改变策略会改变后续状态分布，不能只把动作当静态标签。

`reward_to_go[t]` 是每个动作对应的 `G_t`，也是下一课 REINFORCE 的权重。短有限轨迹结束时使用零 bootstrap；无限 continuing task 则需要价值估计来补上截断之后的尾部，两者目标不同。

In [ ]:
rewards = [1.0, 4.0, 2.0]
rtg = reward_to_go(rewards, gamma=0.9)
print("reward-to-go:", rtg)
assert np.allclose(rtg, [1 + .9*4 + .9**2*2, 4 + .9*2, 2])
assert np.isclose(finite_horizon_returns([1, 4], .9)[-1], 4)
assert np.isclose(finite_horizon_returns([1, 4], .9, bootstrap=10)[-1], 13)

## 4. 练习与边界

练习：把 `long` 的第二步奖励从 4 改为 0，找出两条策略的分界；再手算 `γ=0`。为一个随机转移加入两个 next state，比较期望 Bellman 更新和单次 Q-learning 样本。

这张表没有车辆、延迟、连续油门，也没有证明任何道路性能。它只验证定义和数值关系。下一课把一个很小的动作空间接回真实 MetaDrive 闭环。

In [ ]:
probability_continue = 0.5
mixed = replace(mdp, transitions={**mdp.transitions, "start": {
    **mdp.transitions["start"], "long": ((probability_continue, "long"),
                                        (1 - probability_continue, "terminal"))}})
values = {s: 0.0 for s in mixed.states}
fixed_policy = {"start": "long", "long": "short"}
for _ in range(3):
    values = bellman_policy_update(mixed, values, fixed_policy, gamma=0.9)
expected_target = 1 + 0.9 * probability_continue * 4
rng = np.random.default_rng(7)
sampled_targets = np.where(rng.random(2000) < probability_continue, 4.6, 1.0)
print("Bellman expectation:", values["start"], "sample targets:", sampled_targets[:8],
      "sample mean:", sampled_targets.mean())
assert np.isclose(values["start"], expected_target)

随机练习答案：默认一半概率继续，另一半立即终止，固定long策略的价值为 `1+0.9×0.5×4=2.8`。
一次Q-learning样本的target只能是 `4.6` 或 `1`；已知转移表的Bellman期望是 `2.8`。
样本均值会波动，不能要求单次采样等于期望。现在修改 `probability_continue`，先手算再核对。